# T8: FHVHV impact on traditional taxi services

Analysis of how Uber/Lyft emergence influenced yellow/green taxis and smaller competitors (Via, Juno).
- Absolute trip volumes over time
- Relative market share
- Year-over-year growth
- Pre/post FHVHV comparison

In [ ]:
%%sql -r wh
USE WAREHOUSE BIGDATA_MZMB_WH

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

COLORS = {
    'yellow': '#F5C518',
    'green': '#2ECC71',
    'fhv': '#9B59B6',
    'uber': '#000000',
    'lyft': '#FF00BF',
    'via': '#00CED1',
    'juno': '#FF6347'
}
LABELS = {
    'yellow': 'Yellow taxi',
    'green': 'Green taxi',
    'fhv': 'FHV (other)',
    'uber': 'Uber',
    'lyft': 'Lyft',
    'via': 'Via',
    'juno': 'Juno'
}

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9

def load(name):
    return session.sql(f'SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.{name}').to_pandas()

In [ ]:
monthly = load('T8_MONTHLY_MARKET_SHARE')
monthly['MONTH_DATE'] = pd.to_datetime(monthly['MONTH_DATE'])

fig, ax = plt.subplots(figsize=(14, 6))
for ds in ['yellow', 'green', 'uber', 'lyft', 'fhv', 'via', 'juno']:
    subset = monthly[monthly['DATASET'] == ds].sort_values('MONTH_DATE')
    if len(subset) == 0:
        continue
    lw = 2 if ds in ['yellow', 'green', 'uber', 'lyft'] else 1
    ax.plot(subset['MONTH_DATE'], subset['TRIP_COUNT'] / 1e6,
            color=COLORS[ds], label=LABELS[ds], linewidth=lw, alpha=0.85)

ax.axvline(pd.Timestamp('2019-02-01'), color='orange', linestyle='--', alpha=0.6, label='FHVHV regulation (Feb 2019)')
ax.set_xlabel('Date')
ax.set_ylabel('Trips (millions)')
ax.set_title('Monthly trip volume by operator - absolute comparison')
ax.legend(loc='upper right', framealpha=0.9, fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for ds in ['yellow', 'green', 'uber', 'lyft', 'fhv', 'via', 'juno']:
    subset = monthly[monthly['DATASET'] == ds].sort_values('MONTH_DATE')
    if len(subset) == 0:
        continue
    lw = 2 if ds in ['yellow', 'green', 'uber', 'lyft'] else 1
    ax.plot(subset['MONTH_DATE'], subset['MARKET_SHARE_PCT'],
            color=COLORS[ds], label=LABELS[ds], linewidth=lw, alpha=0.85)

ax.axvline(pd.Timestamp('2019-02-01'), color='orange', linestyle='--', alpha=0.6, label='FHVHV regulation (Feb 2019)')
ax.set_xlabel('Date')
ax.set_ylabel('Market share (%)')
ax.set_title('Monthly market share by operator - relative comparison')
ax.legend(loc='right', framealpha=0.9, fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 80)
plt.tight_layout()
plt.show()

In [ ]:
yearly = load('T8_YEARLY_COMPARISON')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

for ds in ['yellow', 'green', 'uber', 'lyft', 'fhv']:
    subset = yearly[yearly['DATASET'] == ds].sort_values('PICKUP_YEAR')
    if len(subset) == 0:
        continue
    ax1.plot(subset['PICKUP_YEAR'], subset['ANNUAL_TRIPS'] / 1e6,
             color=COLORS[ds], label=LABELS[ds], linewidth=2, marker='o', markersize=4)

ax1.set_xlabel('Year')
ax1.set_ylabel('Annual trips (millions)')
ax1.set_title('Annual trip volume by operator')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

for ds in ['yellow', 'green', 'uber', 'lyft', 'fhv']:
    subset = yearly[yearly['DATASET'] == ds].sort_values('PICKUP_YEAR')
    if len(subset) == 0:
        continue
    ax2.plot(subset['PICKUP_YEAR'], subset['ANNUAL_MARKET_SHARE_PCT'],
             color=COLORS[ds], label=LABELS[ds], linewidth=2, marker='o', markersize=4)

ax2.set_xlabel('Year')
ax2.set_ylabel('Market share (%)')
ax2.set_title('Annual market share by operator')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

years = sorted(yearly['PICKUP_YEAR'].unique())
datasets_stack = ['yellow', 'green', 'fhv', 'uber', 'lyft', 'via', 'juno']
bottom = np.zeros(len(years))

raw_vals = {}
for ds in datasets_stack:
    subset = yearly[yearly['DATASET'] == ds].sort_values('PICKUP_YEAR')
    vals = []
    for yr in years:
        row = subset[subset['PICKUP_YEAR'] == yr]
        val = row['ANNUAL_MARKET_SHARE_PCT'].values[0] if len(row) > 0 else 0
        if ds in ['uber', 'lyft', 'via', 'juno'] and yr < 2019:
            val = 0
        if ds == 'fhv' and yr >= 2019:
            val = 0
        vals.append(val)
    raw_vals[ds] = np.array(vals)

# Normalize each year to 100%
totals = np.zeros(len(years))
for ds in datasets_stack:
    totals += raw_vals[ds]
totals[totals == 0] = 1

for ds in datasets_stack:
    normalized = raw_vals[ds] * 100.0 / totals
    ax.bar(years, normalized, bottom=bottom, color=COLORS[ds], label=LABELS[ds], edgecolor='white', linewidth=0.3)
    bottom += normalized

ax.axvline(2018.5, color='orange', linestyle='--', alpha=0.6, linewidth=1)
ax.text(2018.6, 92, 'FHVHV regulation\n(Feb 2019)', fontsize=7, color='orange')
ax.set_xlabel('Year')
ax.set_ylabel('Market share (%)')
ax.set_title('Stacked market share by year (normalized to 100%)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
yoy = load('T8_YOY_GROWTH')

fig, ax = plt.subplots(figsize=(12, 5))
for ds in ['yellow', 'green', 'uber', 'lyft']:
    subset = yoy[(yoy['DATASET'] == ds) & (yoy['YOY_GROWTH_PCT'].notna())].sort_values('PICKUP_YEAR')
    if len(subset) == 0:
        continue
    ax.plot(subset['PICKUP_YEAR'], subset['YOY_GROWTH_PCT'],
            color=COLORS[ds], label=LABELS[ds], linewidth=2, marker='o', markersize=4)

ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Year-over-year growth (%)')
ax.set_title('Annual growth rate by operator')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
prepost = load('T8_PRE_POST_FHVHV')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

pre_data = prepost[prepost['PRE_FHVHV_AVG_MONTHLY'].notna()].sort_values('PRE_FHVHV_AVG_MONTHLY', ascending=True)
bar_colors = [COLORS.get(d, '#999') for d in pre_data['DATASET']]
bar_labels = [LABELS.get(d, d) for d in pre_data['DATASET']]

x = np.arange(len(pre_data))
width = 0.35
ax1.barh(x - width/2, pre_data['PRE_FHVHV_AVG_MONTHLY'] / 1e6, width, color=bar_colors, alpha=0.7, label='Pre-FHVHV (2015-2018)')
post_vals = pre_data['POST_FHVHV_AVG_MONTHLY'].fillna(0)
ax1.barh(x + width/2, post_vals / 1e6, width, color=bar_colors, alpha=0.35, hatch='//', label='Post-FHVHV (2021-2024)')
ax1.set_yticks(x)
ax1.set_yticklabels(bar_labels)
ax1.set_xlabel('Avg monthly trips (millions)')
ax1.set_title('Traditional services: before vs after FHVHV')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.2, axis='x')

change_data = prepost[prepost['PCT_CHANGE'].notna()].sort_values('PCT_CHANGE')
bar_colors2 = [COLORS.get(d, '#999') for d in change_data['DATASET']]
bar_labels2 = [LABELS.get(d, d) for d in change_data['DATASET']]
ax2.barh(bar_labels2, change_data['PCT_CHANGE'], color=bar_colors2, edgecolor='black', linewidth=0.3)
ax2.axvline(0, color='black', linewidth=0.5)
ax2.set_xlabel('% change')
ax2.set_title('Percentage change in avg monthly trips (pre vs post FHVHV)')
ax2.grid(True, alpha=0.2, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

small = monthly[monthly['DATASET'].isin(['via', 'juno'])].copy()
for ds in ['via', 'juno']:
    subset = small[small['DATASET'] == ds].sort_values('MONTH_DATE')
    if len(subset) == 0:
        continue
    ax.plot(subset['MONTH_DATE'], subset['TRIP_COUNT'] / 1e6,
            color=COLORS[ds], label=LABELS[ds], linewidth=2, marker='o', markersize=3)

ax.set_xlabel('Date')
ax.set_ylabel('Trips (millions)')
ax.set_title('Smaller FHVHV competitors: Via and Juno (absolute trips)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nJuno (HV0002) ceased operations in late 2019 after being acquired by Gett.")
print("Via (HV0004) peaked in 2019 at ~1M trips/month, dropped during COVID,")
print("and has not recovered to pre-pandemic levels.")